In [34]:
import os
import time
import torch
import librosa
import numpy as np
import itertools
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.tensorboard import SummaryWriter
from datasets import load_dataset, get_dataset_config_names
from transformers import Wav2Vec2Model

# --- Industry Standard Directory Logic ---
# Assuming notebook is in 'notebooks/' and we save to root
ROOT_DIR = Path(os.getcwd()).parent
MODELS_DIR = ROOT_DIR / "models"
LOGS_DIR = ROOT_DIR / "logs"

for d in [MODELS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [5]:
class AudioToBlendshapeModel(torch.nn.Module):
    def __init__(self, num_blendshapes=52, hidden_dim=512, use_pretrained=False):
        super().__init__()
        self.use_pretrained = use_pretrained
        
        if self.use_pretrained:
            # Mode A: Pre-trained Phonetic Encoder (Wav2Vec 2.0)
            self.audio_backbone = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")
            self.audio_proj = torch.nn.Linear(768, hidden_dim)
        else:
            # Mode B: Custom Phonetic-CNN (No pre-training)
            self.audio_backbone = torch.nn.Sequential(
                torch.nn.Conv1d(1, 64, kernel_size=10, stride=5),
                torch.nn.BatchNorm1d(64),
                torch.nn.ReLU(),
                torch.nn.Conv1d(64, 128, kernel_size=3, stride=2),
                torch.nn.ReLU(),
                torch.nn.Conv1d(128, 256, kernel_size=3, stride=2),
                torch.nn.ReLU(),
                torch.nn.Conv1d(256, hidden_dim, kernel_size=3, stride=2),
                torch.nn.ReLU()
            )

        encoder_layer = torch.nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=8, batch_first=True
        )
        self.transformer = torch.nn.TransformerEncoder(encoder_layer, num_layers=4)
        
        self.regressor = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim, 256),
            torch.nn.LeakyReLU(0.2),
            torch.nn.Linear(256, num_blendshapes),
            torch.nn.Sigmoid() 
        )

    def forward(self, x):
        if self.use_pretrained:
            # Wav2Vec expects (Batch, Samples)
            features = self.audio_backbone(x).last_hidden_state
            x = self.audio_proj(features)
        else:
            # CNN expects (Batch, Channels, Samples)
            if x.dim() == 2: x = x.unsqueeze(1)
            x = self.audio_backbone(x).transpose(1, 2)
        
        x = self.transformer(x)
        return self.regressor(x)

In [22]:
class SyntheticBEATDataset(Dataset):
    def __init__(self, num_samples=100):
        self.num_samples = num_samples
        self.sr = 16000 # 16kHz

    def __len__(self): 
        return self.num_samples

    def __getitem__(self, idx):
        # 2 seconds of audio
        audio = torch.randn(self.sr * 2) 
        # 2 seconds of blendshapes at 60fps (120 frames)
        blendshapes = torch.rand(120, 52) 
        return audio, blendshapes

def collate_fn(batch):
    audio_seqs, bs_seqs = zip(*batch)
    audio_padded = pad_sequence(audio_seqs, batch_first=True)
    bs_padded = pad_sequence(bs_seqs, batch_first=True)
    return audio_padded, bs_padded

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for audio, target in loader:
        audio, target = audio.to(device), target.to(device)
        optimizer.zero_grad()
        
        # 1. Forward Pass
        output = model(audio) # output shape: [Batch, Seq_Len_A, 52]
        
        # 2. Alignment: Interpolate output to match target's temporal length
        # target shape: [Batch, Seq_Len_B, 52] -> usually 120
        if output.shape[1] != target.shape[1]:
            # Interpolate expects [Batch, Channels, Length]
            output = output.transpose(1, 2) 
            output = F.interpolate(output, size=target.shape[1], mode='linear', align_corners=False)
            output = output.transpose(1, 2)

        # 3. Standard Loss
        mse = criterion(output, target)
        
        # Velocity Loss (Smoothness)
        vel_loss = criterion(output[:, 1:] - output[:, :-1], target[:, 1:] - target[:, :-1])
        
        loss = mse + 0.5 * vel_loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    val_loss = 0
    total_mae = 0
    with torch.no_grad():
        for audio, target in loader:
            audio, target = audio.to(device), target.to(device)
            output = model(audio)
            
            # Match lengths
            if output.shape[1] != target.shape[1]:
                output = output.transpose(1, 2)
                output = F.interpolate(output, size=target.shape[1], mode='linear', align_corners=False)
                output = output.transpose(1, 2)
                
            val_loss += criterion(output, target).item()
            total_mae += F.l1_loss(output, target).item()
    return val_loss / len(loader), total_mae / len(loader)

def run_trial(config, train_loader, val_loader):
    run_id = f"TRIAL_{int(time.time())}_pre{config['use_pretrained']}_h{config['hidden_dim']}"
    writer = SummaryWriter(log_dir=LOGS_DIR / run_id)
    
    model = AudioToBlendshapeModel(
        hidden_dim=config['hidden_dim'], 
        use_pretrained=config['use_pretrained']
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
    criterion = torch.nn.MSELoss()
    best_val_loss = float('inf')
    
    for epoch in range(config['epochs']):
        t_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        v_loss, v_mae = validate(model, val_loader, criterion, device)
        
        print(f"Epoch {epoch+1}/{config['epochs']} -> Train Loss: {t_loss:.6f}, Val Loss: {v_loss:.6f}, Val MAE: {v_mae:.6f}")
        
        writer.add_scalars("Loss/Combined", {"train": t_loss, "val": v_loss}, epoch)
        writer.add_scalar("Accuracy/Val_MAE", v_mae, epoch)
        
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            torch.save({
                'model_state_dict': model.state_dict(),
                'config': config,
                'val_loss': best_val_loss
            }, MODELS_DIR / f"{run_id}_best.pt")
            
    writer.add_hparams(config, {"hparam/best_val_loss": best_val_loss})
    writer.close()
    return best_val_loss

In [36]:
# 1. Setup Search Space
search_space = {
    "lr": [1e-4, 5e-5],
    "hidden_dim": [256, 512],
    "use_pretrained": [True, False],
    "batch_size": [4],
    "epochs": [5]
}

# 2. Setup Synthetic Data
dataset = SyntheticBEATDataset(num_samples=50)
train_set, val_set = random_split(dataset, [40, 10])

# 3. Execution
keys, values = zip(*search_space.items())
trials = [dict(zip(keys, v)) for v in itertools.product(*values)]

print(f"Total Trials to Run: {len(trials)}")

for i, cfg in enumerate(trials):
    print(f"\n--- [Trial {i+1}/{len(trials)}] Config: {cfg} ---")
    t_loader = DataLoader(train_set, batch_size=cfg['batch_size'], shuffle=True, collate_fn=collate_fn)
    v_loader = DataLoader(val_set, batch_size=cfg['batch_size'], shuffle=False, collate_fn=collate_fn)
    
    try:
        res = run_trial(cfg, t_loader, v_loader)
        print(f"Success. Best Val Loss: {res:.6f}")
    except Exception as e:
        print(f"Trial crashed: {e}")

Total Trials to Run: 8

--- [Trial 1/8] Config: {'lr': 0.0001, 'hidden_dim': 256, 'use_pretrained': True, 'batch_size': 4, 'epochs': 5} ---


Loading weights: 100%|██████████| 210/210 [00:00<00:00, 2721.63it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Success. Best Val Loss: inf

--- [Trial 2/8] Config: {'lr': 0.0001, 'hidden_dim': 256, 'use_pretrained': False, 'batch_size': 4, 'epochs': 5} ---
Success. Best Val Loss: 0.082931

--- [Trial 3/8] Config: {'lr': 0.0001, 'hidden_dim': 512, 'use_pretrained': True, 'batch_size': 4, 'epochs': 5} ---


Loading weights: 100%|██████████| 210/210 [00:00<00:00, 2179.35it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Success. Best Val Loss: inf

--- [Trial 4/8] Config: {'lr': 0.0001, 'hidden_dim': 512, 'use_pretrained': False, 'batch_size': 4, 'epochs': 5} ---


KeyboardInterrupt: 